# Check arXiv astro-ph RSS Feed

AstroCoffee utility to interroate the current astro-ph RSS feed on arXiv when there are problems.

The core code is the same code as used by the `grind.py` CGI web script, except that it only does the
download and parse steps without creating the paper selection webform.

It prints a few diagnostics of what it found, and will show info on the first paper it finds in the feed,
if it finds any papers at all.

The RSS feed URL is `arxiv.org/rss/astro-ph`, which has no date specification, and there is no archive of 
previous RSS feed XML files.  What you see at this URL is what it made that day.

The date coding inside the RSS feed is the `pubDate` tag which lists the publication date of each paper in
the feed (one pubDate per paper).  One diagnostic is to show `pubDate` for the first and last papers.

The other is to show the info for the first and last papers (title, authors, etc.) that would be shown
in the paper selection form.  This is a good way to test if the day's RSS feed glitch
is that a previous posting's papers are in the new posting's RSS feed.

There is nothing we can do if the data in the RSS feed they give us is bad.

In [12]:
import sys
import os
import re
import numpy as np
from datetime import datetime, timezone

# XML parsing

import requests
from bs4 import BeautifulSoup

# HTML display inside the notebook

from IPython.display import HTML


## RSS Feed URL etc.

This also prints the current local and UTC date tags used by elements of the AstroCoffee database.

In [13]:
versionID = '3.0.5'
versionDate = '2026-03-08'

# ArXiV RSS Feed for astro-ph

rssFeed = 'http://arxiv.org/rss/astro-ph'

# isoTime

now = datetime.now()
calDate = now.strftime('%A %B %e')
print(f"Today's local date is {calDate}")

utc = datetime.now(timezone.utc)
isoDate = utc.strftime('%Y%b%d')
print(f"Today's UTC date is {isoDate}")

# daily paper database file

coffeePot = './'  # else /var/www/CoffeePot

dbFile = f'{coffeePot}/astro-ph.pdb'

# maximum number of authors to list before switching to et al.

maxAuth = 10


Today's local date is Tuesday March 10
Today's UTC date is 2026Mar10


## Read the astro-ph RSS feed

In [14]:
r = requests.get(rssFeed)
soup = BeautifulSoup(r.text,'xml')
papers = soup.findAll('item')

numPapers = len(papers)

# does the feed contain any papers?

print(f"Today's RSS Feed contains {numPapers} papers")

Today's RSS Feed contains 122 papers


### Parse the RSS feed response into data we need

 * `paperID` = paper's arXiv ID (e.g., 2401.17318)
 * `absURL` = URL of the arXiv full abstract page
 * `pdfURL` = URL of the paper PDF file
 * `paperName` = unique paper identifier (e.g., arxiv:2401.17318)
 * `paperType` = type of submission (NEW, CROSS, REPLACE)
 * `paperTitle` = full text of the paper title.
 * `paperAuths` = abbreviated paper list
 * `paperAbst` = full text of the paper abstract

These get stored in a paper database file for use by CGI scripts downstream.  This makes sure we only 
make 1 (or at least very few) queries of the RSS feed so an over-enthusiastic user doesn't hammer the
arxiv feed and get tagged (incorrectly) as a bot.

In [15]:
# posting date derived from the pubData element

pubDate = soup.findAll('pubDate')
pubBits = pubDate[0].text.split(' ')
postingDate = f'{pubBits[3]} {pubBits[2]} {pubBits[1]}'

# temporary lists because of some infelicities in the feed

arxivIDs = []
titles = []
absLinks = []
pdfLinks = []
absNames = []
abstracts = []
authors = []
subTypes = []

for paper in papers:
    titles.append(paper.find('title').text)
    link = paper.find('link').text
    
    # pick apart the link into the bits we need
    
    protocol,url = link.split('//')
    urlBits = url.split('/')
    paperID = urlBits[-1]
    arxivIDs.append(paperID)
    absLinks.append(f'{protocol}//arxiv.org/abs/{paperID}')
    pdfLinks.append(f'{protocol}//arxiv.org/pdf/{paperID}.pdf')
    absNames.append(f'arXiv:{paperID}')
    
    # submission type
    
    subTypes.append(paper.find('arxiv:announce_type').text.upper())

    # paper abstract
    
    abstracts.append(paper.find('description').text)

    # authors - broken into multiple elements, merge...

    rawAuth = paper.find('dc:creator')
    pAuth = re.sub( "<\/?dc:creator>", "", rawAuth.text).split(',')

    for auth in pAuth:
        if auth == pAuth[0]:
            fullAuth = pAuth[0]
        else:
            fullAuth = f'{fullAuth}, {auth}'
    if len(pAuth) > 1:
        authStr = f'{pAuth[0]}, et al. [{len(pAuth)-1} co-authors]'
    else:
        authStr = f'{pAuth}'
        fullAuth = pAuth

    
    if len(pAuth) > maxAuth or len(pAuth) == 1:
        authors.append(authStr)
    else:
        authors.append(fullAuth)

    
# numpy arrays of our primary data

paperID = np.array(arxivIDs)
absURL = np.array(absLinks)
pdfURL = np.array(pdfLinks)
paperName = np.array(absNames)
paperType = np.array(subTypes)
paperTitle = np.array(titles)
paperAuths = np.array(authors)
paperAbst = np.array(abstracts)

# accounting

numPapers = len(papers)

# number of new submissions

iNew = np.where(paperType=='NEW')[0]
numNew = len(iNew)

# number of cross-posted papers

iCross = np.where(paperType=='CROSS')[0]
numCross = len(iCross)

# number of replacements

iUpdated = np.where((paperType=='REPLACE') | (paperType=='REPLACE-CROSS'))[0]
numUpdated = len(iUpdated)

# quick report

print(f"astro-ph RSS Feed summary:")
print(f"  Found {numPapers} papers on astro-ph today:")
print(f"     {numNew} new")
print(f"     {numCross} cross-posted")
print(f"     {numUpdated} replacements\n")
print(f"  First pubDate tag: {pubDate[0]}")
print(f"     Paper: {paperName[0]}")
print(f"     Title: {paperTitle[0]}\n")
print(f"   Last pubDate tag: {pubDate[-1]}")
print(f"     Paper: {paperName[-1]}")
print(f"     Title: {paperTitle[-1]}\n")


astro-ph RSS Feed summary:
  Found 122 papers on astro-ph today:
     73 new
     12 cross-posted
     37 replacements

  First pubDate tag: <pubDate>Tue, 10 Mar 2026 00:00:00 -0400</pubDate>
     Paper: arXiv:2603.05571
     Title: UK White Paper on Magnetohydrodynamic (MHD) seismology of solar and heliospheric plasmas

   Last pubDate tag: <pubDate>Tue, 10 Mar 2026 00:00:00 -0400</pubDate>
     Paper: arXiv:2601.13532
     Title: An upper limit on cosmological chiral gravitational wave background

